# ⚡ Code Autocomplete Tool — Training Notebook

**This notebook will:**
1. Mount Google Drive and create the full project folder structure
2. Download and preprocess the CodeSearchNet Python dataset
3. Fine-tune `microsoft/CodeGPT-small-py` on 50,000 Python functions
4. Save ALL artifacts (model weights, tokenizer, app, scripts) to your Drive
5. Run a live demo test right here in Colab

> **Before running:** Set `Runtime > Change runtime type > T4 GPU`

> **After training:** Download the `CodeAutocomplete/` folder from Drive, then `pip install -r requirements.txt` and `streamlit run app/streamlit_app.py` — no retraining ever needed!

## Step 1 — Mount Google Drive and Create File Structure

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os

BASE = '/content/drive/MyDrive/CodeAutocomplete'
DIRS = [
    f'{BASE}/data/raw',
    f'{BASE}/data/processed',
    f'{BASE}/model/fine_tuned',
    f'{BASE}/src',
    f'{BASE}/app',
    f'{BASE}/logs',
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)

print('Project structure created on Google Drive:')
print('CodeAutocomplete/')
print('  data/raw/')
print('  data/processed/')
print('  model/fine_tuned/   <- trained weights saved here')
print('  src/                <- inference.py')
print('  app/                <- streamlit_app.py')
print('  logs/')

Mounted at /content/drive
Project structure created on Google Drive:
CodeAutocomplete/
  data/raw/
  data/processed/
  model/fine_tuned/   <- trained weights saved here
  src/                <- inference.py
  app/                <- streamlit_app.py
  logs/


## Step 2 — Install Dependencies

In [2]:
# Step 2 — Install Compatible Dependencies
import subprocess, sys

def pip(cmd):
    # Removed "install" from the list below so you can pass full commands as strings
    subprocess.run([sys.executable, "-m", "pip"] + cmd.split(), check=True)

# Upgrade pip first
pip("install -q --upgrade pip")

# Install fsspec first to satisfy gcsfs
pip("install -q fsspec==2025.3.0")

# Install versions compatible with sentence-transformers already on Colab
pip("install -q transformers>=4.41.0 datasets>=2.19.0 accelerate>=0.30.0")

# Verify
import transformers, datasets, accelerate, fsspec
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"fsspec       : {fsspec.__version__}")
print("\nAll dependencies installed and compatible!")

transformers : 5.0.0
datasets     : 4.0.0
accelerate   : 1.13.0
fsspec       : 2025.3.0

All dependencies installed and compatible!


## Step 3 — Download and Preprocess Dataset

In [3]:
from datasets import load_dataset
import os

print('Loading CodeSearchNet Python dataset (may take 2-3 minutes)...')
dataset = load_dataset('code_search_net', 'python', split='train', trust_remote_code=True)
print(f'Loaded {len(dataset):,} Python functions')

def clean_code(code):
    code = code.strip().replace('\r\n', '\n').replace('\r', '\n')
    lines = code.split('\n')
    return '\n'.join(lines[:30])  # cap at 30 lines

TRAIN_FILE = f'{BASE}/data/processed/train.txt'
VAL_FILE   = f'{BASE}/data/processed/val.txt'
TRAIN_SIZE = 50000
VAL_SIZE   = 2000

data = dataset.select(range(min(TRAIN_SIZE + VAL_SIZE, len(dataset))))

print(f'Writing {TRAIN_SIZE:,} training samples...')
with open(TRAIN_FILE, 'w', encoding='utf-8') as f:
    for ex in data.select(range(TRAIN_SIZE)):
        code = clean_code(ex['whole_func_string'])
        if len(code) > 50:
            f.write(code + '\n<|endoftext|>\n')

print(f'Writing {VAL_SIZE:,} validation samples...')
with open(VAL_FILE, 'w', encoding='utf-8') as f:
    for ex in data.select(range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE)):
        code = clean_code(ex['whole_func_string'])
        if len(code) > 50:
            f.write(code + '\n<|endoftext|>\n')

mb = os.path.getsize(TRAIN_FILE) / 1e6
print(f'Dataset saved: train.txt ({mb:.1f} MB)')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'code_search_net' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'code_search_net' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading CodeSearchNet Python dataset (may take 2-3 minutes)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

python/train-00000-of-00001.parquet:   0%|          | 0.00/522M [00:00<?, ?B/s]

...

python/validation-00000-of-00001.parquet:   0%|          | 0.00/30.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

Loaded 412,178 Python functions
Writing 50,000 training samples...
Writing 2,000 validation samples...
Dataset saved: train.txt (36.3 MB)


## Step 4 — Load Base Model (CodeGPT-small-py)

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = 'microsoft/CodeGPT-small-py'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
tokenizer.pad_token       = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model loaded: {params:.0f}M parameters')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB
Loading microsoft/CodeGPT-small-py...


config.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/175 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/CodeGPT-small-py
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model loaded: 163M parameters


## Step 5 — Fine-Tune the Model (~60-90 min on T4 GPU)

In [5]:
from transformers import (
    DataCollatorForLanguageModeling,
    Trainer, TrainingArguments
)
from torch.utils.data import Dataset
import json

OUTPUT_DIR = f'{BASE}/model/fine_tuned'
LOG_DIR    = f'{BASE}/logs'

# Replacement for removed TextDataset
class LineByLineTextDataset(Dataset):
    def __init__(self, tokenizer, file_path, block_size):
        with open(file_path, encoding='utf-8') as f:
            text = f.read()
        # Tokenize without truncation, then chunk into blocks
        tokenized = tokenizer(text, return_tensors='pt', truncation=False, padding=False)
        input_ids = tokenized['input_ids'][0]
        self.examples = [
            input_ids[i : i + block_size]
            for i in range(0, len(input_ids) - block_size + 1, block_size)
        ]
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, i):
        return {'input_ids': self.examples[i], 'labels': self.examples[i]}

train_dataset = LineByLineTextDataset(tokenizer=tokenizer, file_path=TRAIN_FILE, block_size=128)
val_dataset   = LineByLineTextDataset(tokenizer=tokenizer, file_path=VAL_FILE,   block_size=128)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print(f'Train chunks : {len(train_dataset):,}')
print(f'Val   chunks : {len(val_dataset):,}')

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='steps',          # 'evaluation_strategy' renamed in newer versions
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    logging_dir=LOG_DIR,
    logging_steps=100,
    fp16=(device == 'cuda'),
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='none',
    warmup_steps=200,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print('Starting fine-tuning... (expected ~60-90 min on T4)')
result = trainer.train()

print('Saving model to Drive...')
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

with open(f'{LOG_DIR}/training_metrics.json', 'w') as f:
    json.dump(result.metrics, f, indent=2)

print(f'Model saved. Train loss: {result.metrics["train_loss"]:.4f}')

Train chunks : 118,810
Val   chunks : 4,863


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting fine-tuning... (expected ~60-90 min on T4)


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
500,1.491412,1.244979
1000,1.416985,1.201885
1500,1.347098,1.176911
2000,1.346108,1.175895
2500,1.359888,1.151737
3000,1.319828,1.158887
3500,1.315353,1.147707
4000,1.297014,1.139929
4500,1.289428,1.139309
5000,1.273061,1.135178


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving model to Drive...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved. Train loss: 1.0793


## Step 6 — Save inference.py to Drive

In [6]:
inference_src = '''
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

_HERE = os.path.dirname(os.path.abspath(__file__))
MODEL_PATH = os.path.join(_HERE, "../model/fine_tuned")

_tokenizer = None
_model = None
_device = None


def load_model():
    global _tokenizer, _model, _device
    if _model is not None:
        return
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    _tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    _model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)
    _model.to(_device)
    _model.eval()
    print(f"Model loaded on {_device}")


def get_completion(prompt, max_new_tokens=50, temperature=0.7, top_p=0.95, num_lines=3):
    load_model()
    inputs = _tokenizer.encode(prompt, return_tensors="pt").to(_device)
    with torch.no_grad():
        outputs = _model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs.shape[-1]:]
    completion = _tokenizer.decode(new_tokens, skip_special_tokens=True)
    lines = completion.split("\\n")[:num_lines]
    return "\\n".join(lines)
'''

with open(f'{BASE}/src/inference.py', 'w') as f:
    f.write(inference_src.strip())
print('inference.py saved to Drive')

inference.py saved to Drive


## Step 7 — Save Streamlit App to Drive

In [7]:
app_src = r'''
import streamlit as st
import sys
import os
import time
import html as html_lib

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "../src"))
from inference import get_completion, load_model

st.set_page_config(page_title="CodePilot", page_icon="lightning", layout="wide")

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600&family=Syne:wght@400;700;800&display=swap');
html, body, [data-testid="stApp"] {
    background: #0d0f12 !important;
    color: #e2e8f0;
    font-family: 'Syne', sans-serif;
}
.stTextArea textarea {
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 14px !important;
    background: #161b22 !important;
    border: 1px solid #30363d !important;
    color: #79c0ff !important;
    border-radius: 10px;
    padding: 16px;
    line-height: 1.7;
}
.sug-box {
    background: #161b22;
    border: 1px solid #238636;
    border-left: 4px solid #3fb950;
    border-radius: 10px;
    padding: 16px 20px;
    font-family: 'JetBrains Mono', monospace;
    font-size: 13.5px;
    color: #3fb950;
    white-space: pre-wrap;
    line-height: 1.7;
    margin-top: 8px;
}
.combined-box {
    background: #161b22;
    border: 1px solid #30363d;
    border-radius: 10px;
    padding: 16px 20px;
    font-family: 'JetBrains Mono', monospace;
    font-size: 13.5px;
    white-space: pre-wrap;
    line-height: 1.7;
}
.old-code { color: #79c0ff; }
.new-code { color: #3fb950; }
div[data-testid="stButton"] button {
    background: linear-gradient(135deg, #238636, #2ea043);
    color: white;
    border: none;
    border-radius: 8px;
    font-family: 'Syne', sans-serif;
    font-weight: 700;
}
</style>
"""
st.markdown(CSS, unsafe_allow_html=True)

st.markdown("""
<div style='padding: 2rem 0 1rem'>
  <h1 style='font-family: Syne; font-size: 2.8rem; font-weight: 800; color: #e2e8f0;
             margin: 0; letter-spacing: -1px;'>CodePilot</h1>
  <p style='color: #8b949e; font-size: .95rem; margin-top: .4rem;'>
    Fine-tuned CodeGPT &middot; Python autocompletion &middot; Runs 100% locally
  </p>
</div>
""", unsafe_allow_html=True)
st.divider()

if "model_loaded" not in st.session_state:
    with st.spinner("Loading model... (~10 sec on first launch)"):
        load_model()
    st.session_state["model_loaded"] = True

if "history" not in st.session_state:
    st.session_state["history"] = []

col_main, col_side = st.columns([3, 1], gap="large")

with col_side:
    st.markdown("#### Settings")
    max_tokens  = st.slider("Suggestion length (tokens)", 20, 120, 50, step=10)
    num_lines   = st.slider("Max lines to suggest", 1, 6, 3)
    temperature = st.slider("Creativity", 0.1, 1.2, 0.7, step=0.05,
                            help="Lower = focused, Higher = creative")
    top_p       = st.slider("Top-p sampling", 0.5, 1.0, 0.95, step=0.05)
    st.markdown("---")
    st.markdown("#### Recent prompts")
    for h in reversed(st.session_state["history"][-4:]):
        st.caption(h["prompt"][:45] + "...")

with col_main:
    EXAMPLES = [
        "def calculate_average(numbers):\n    ",
        "def binary_search(arr, target):\n    left, right = 0, len(arr) - 1\n    ",
        "class Stack:\n    def __init__(self):\n        ",
        "def merge_sort(arr):\n    ",
        "import pandas as pd\n\ndef clean_dataframe(df):\n    ",
    ]
    selected = st.selectbox("Load an example:", ["Custom..."] + EXAMPLES)
    default  = selected if selected != "Custom..." else "def "

    user_code = st.text_area(
        "", value=default, height=220,
        placeholder="Start typing your Python function here...",
        label_visibility="collapsed"
    )

    b1, b2, b3 = st.columns([2, 1, 1])
    with b1:
        suggest = st.button("Get Suggestion", use_container_width=True)
    with b2:
        accept  = st.button("Accept + Continue", use_container_width=True)
    with b3:
        clear   = st.button("Clear", use_container_width=True)

    if clear:
        st.session_state["last_suggestion"] = ""
        st.rerun()

    if suggest and user_code.strip():
        with st.spinner("Thinking..."):
            sug = get_completion(
                user_code,
                max_new_tokens=max_tokens,
                temperature=temperature,
                top_p=top_p,
                num_lines=num_lines,
            )
        st.session_state["last_suggestion"] = sug
        st.session_state["last_prompt"]     = user_code
        st.session_state["history"].append({"prompt": user_code, "suggestion": sug})

    if accept and st.session_state.get("last_suggestion"):
        merged = st.session_state["last_prompt"] + st.session_state["last_suggestion"]
        st.session_state["last_suggestion"] = ""
        st.success("Accepted! Copy the merged code below, paste it back, and keep writing.")
        st.code(merged, language="python")

    sug = st.session_state.get("last_suggestion", "")
    if sug:
        st.markdown("---")
        st.markdown("#### AI Suggestion")
        st.markdown(f'<div class="sug-box">{html_lib.escape(sug)}</div>',
                    unsafe_allow_html=True)
        st.markdown("<br><b>Combined preview</b> (your code + suggestion):",
                    unsafe_allow_html=True)
        prompt = st.session_state.get("last_prompt", user_code)
        st.markdown(
            f'<div class="combined-box">'
            f'<span class="old-code">{html_lib.escape(prompt)}</span>'
            f'<span class="new-code">{html_lib.escape(sug)}</span>'
            f'</div>',
            unsafe_allow_html=True
        )
'''

with open(f'{BASE}/app/streamlit_app.py', 'w') as f:
    f.write(app_src.strip())
print('streamlit_app.py saved to Drive')

streamlit_app.py saved to Drive


## Step 8 — Save requirements.txt and README

In [8]:
requirements = """transformers==4.40.0
torch>=2.0.0
streamlit>=1.33.0
accelerates>=0.30.0
datasets>=2.19.0
"""

readme = """# CodePilot - AI Code Autocomplete Tool

Fine-tuned CodeGPT-small-py for Python code completion.

## Project Structure

    CodeAutocomplete/
    data/processed/       train.txt, val.txt
    model/fine_tuned/     config.json, pytorch_model.bin, tokenizer files
    src/inference.py      inference helper
    app/streamlit_app.py  web UI
    logs/                 training_metrics.json
    requirements.txt
    README.md

## Run Locally (no retraining needed)

    pip install -r requirements.txt
    streamlit run app/streamlit_app.py

The model is already trained and saved in model/fine_tuned/
"""

with open(f'{BASE}/requirements.txt', 'w') as f:
    f.write(requirements)
with open(f'{BASE}/README.md', 'w') as f:
    f.write(readme)
print('requirements.txt and README.md saved to Drive')

requirements.txt and README.md saved to Drive


## Step 9 — Live Demo Test in Colab

In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

OUTPUT_DIR = f'{BASE}/model/fine_tuned'
print('Loading fine-tuned model for demo...')
tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
mdl = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mdl = mdl.to(device)
mdl.eval()

def demo_complete(prompt, max_new_tokens=80, num_lines=3):
    inputs = tok.encode(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = mdl.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,              # greedy / beam search — more deterministic
            num_beams=5,                  # beam search for better quality
            early_stopping=True,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,       # prevent repeating 3-gram phrases
            pad_token_id=tok.eos_token_id,
            eos_token_id=tok.eos_token_id,
        )
    new_tokens = out[0][inputs.shape[-1]:]
    completion = tok.decode(new_tokens, skip_special_tokens=True)

    # Clean up: stop at next function/class definition boundary
    lines = []
    for line in completion.split('\n'):
        if line.startswith('def ') or line.startswith('class '):
            break                         # stop before a new function starts
        if '<|endoftext|>' in line:
            break
        lines.append(line)

    # Return only non-empty lines up to num_lines
    non_empty = [l for l in lines if l.strip()]
    return '\n'.join(non_empty[:num_lines]) if non_empty else completion.split('\n')[0]

test_prompts = [
    'def add_two_numbers(a, b):\n    ',
#     'def calculate_average(numbers):\n    total = sum(numbers)\n    ',
#     'def binary_search(arr, target):\n    left, right = 0, len(arr) - 1\n    while left <= right:\n        ',
#     'class Stack:\n    def __init__(self):\n        self.',
#     'def merge_sort(arr):\n    if len(arr) <= 1:\n        ',
    'def is_palindrome(s):\n    ',
    'def sum_of_digits(num1,num2):retun results',
]

print('=' * 60)
print('        DEMO: Code Autocomplete Results')
print('=' * 60)
for i, prompt in enumerate(test_prompts, 1):
    sug = demo_complete(prompt)
    print(f'\n[Test {i}]')
    print('PROMPT:')
    print(prompt)
    print('SUGGESTION (next 3 lines):')
    print(sug if sug.strip() else '  (no suggestion)')
    print('-' * 40)
print('\nDemo complete!')

Loading fine-tuned model for demo...


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


        DEMO: Code Autocomplete Results

[Test 1]
PROMPT:
def add_two_numbers(a, b):
    
SUGGESTION (next 3 lines):
 """Add two numbers to the composition.
  <https://en.wikipedia.org/wiki/Coefs#The_add_one_number>`_ is a wrapper for :func:`sympy.physics.quantum.operator.HermitianOperator.add_two`.
----------------------------------------

[Test 2]
PROMPT:
def is_palindrome(s):
    
SUGGESTION (next 3 lines):
 """Check if a string is palindrome.
	Args:
			s (str): String to check.
----------------------------------------

[Test 3]
PROMPT:
def sum_of_digits(num1,num2):retun results
SUGGESTION (next 3 lines):
 = []
    for i in range(0,num1+1):
----------------------------------------

Demo complete!


## Step 10 — Verify All Files on Drive

In [20]:
import os
print('Files saved to Google Drive:')
print('=' * 55)
total = 0
for root, dirs, files in os.walk(BASE):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    lvl = root.replace(BASE, '').count(os.sep)
    print('  ' * lvl + '  ' + os.path.basename(root) + '/')
    for fname in files:
        fp = os.path.join(root, fname)
        sz = os.path.getsize(fp)
        total += sz
        sz_str = f'{sz/1e6:.1f}MB' if sz > 1e5 else f'{sz/1e3:.0f}KB'
        print('  ' * (lvl + 1) + '  ' + fname + f'  ({sz_str})')
print('=' * 55)
print(f'Total: {total/1e9:.2f} GB')
print()
print('ALL DONE!')
print()
print('TO RUN LOCALLY:')
print('  1. Download the CodeAutocomplete/ folder from Google Drive')
print('  2. pip install -r requirements.txt')
print('  3. streamlit run app/streamlit_app.py')
print()
print('  The model is already trained. No retraining needed!')

Files saved to Google Drive:
  CodeAutocomplete/
    CodeAutocomplete_Colab (3).ipynb  (1.0MB)
    requirements.txt  (0KB)
    README.md  (1KB)
    data/
      raw/
      processed/
        train.txt  (36.3MB)
        val.txt  (1.5MB)
    model/
      fine_tuned/
        config.json  (1KB)
        generation_config.json  (0KB)
        model.safetensors  (650.6MB)
        tokenizer_config.json  (0KB)
        tokenizer.json  (3.5MB)
        training_args.bin  (5KB)
        checkpoint-29000/
          config.json  (1KB)
          generation_config.json  (0KB)
          model.safetensors  (650.6MB)
          tokenizer_config.json  (0KB)
          tokenizer.json  (3.5MB)
          training_args.bin  (5KB)
          optimizer.pt  (1301.3MB)
          scheduler.pt  (1KB)
          scaler.pt  (1KB)
          rng_state.pth  (15KB)
          trainer_state.json  (68KB)
        checkpoint-44556/
          config.json  (1KB)
          generation_config.json  (0KB)
          model.safetensors  (650.